# Global Happiness & Well Being Analysis


## Data Cleaning, Inspection and Imputation

### Data Inspection (Raw Annual Files)

Reviewing shape metrics and column headers for **2017**, **2018**, and **2019** raw CSVs to identify potential schema drifts prior to concatenation.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import country_converter as coco

df1=pd.read_csv("../data/raw/2017.csv")
df2=pd.read_csv("../data/raw/2018.csv")
df3=pd.read_csv("../data/raw/2019.csv")

files=[df1,df2,df3]
for file in files:
    rc=file.shape
    print("=" * 20,"Number Of Rows and Columns","="*20)
    print(f"The file contains {rc[0]} rows and {rc[1]} columns")
    print("\n","=" * 20,"Names Of Columns","="*20)
    print(file.columns)

==================== Number Of Rows and Columns ====================
The file contains 155 rows and 12 columns

 ==================== Names Of Columns ====================
Index(['Country', 'Happiness.Rank', 'Happiness.Score', 'Whisker.high',
       'Whisker.low', 'Economy..GDP.per.Capita.', 'Family',
       'Health..Life.Expectancy.', 'Freedom', 'Generosity',
       'Trust..Government.Corruption.', 'Dystopia.Residual'],
      dtype='str')
==================== Number Of Rows and Columns ====================
The file contains 156 rows and 9 columns

 ==================== Names Of Columns ====================
Index(['Overall rank', 'Country or region', 'Score', 'GDP per capita',
       'Social support', 'Healthy life expectancy',
       'Freedom to make life choices', 'Generosity',
       'Perceptions of corruption'],
      dtype='str')
==================== Number Of Rows and Columns ====================
The file contains 156 rows and 9 columns

 ==================== Names Of Columns ===

### Column Renaming & Feature Alignment

Renaming columns across `df1` (2017), `df2` (2018), and `df3` (2019) to ensure seamless row wise concatenation (`pd.concat`) without introducing duplicate or `NaN` columns.

In [2]:
df1=df1.rename(columns={'Happiness.Rank':'Overall rank','Happiness.Score':'Score','Economy..GDP.per.Capita.':'GDP per capita','Health..Life.Expectancy.':'Healthy life expectancy','Trust..Government.Corruption.':'Perceptions of corruption','Family':'Social support'})
df2=df2.rename(columns={'Freedom to make life choices':'Freedom','Country or region':'Country'})
df3=df3.rename(columns={'Freedom to make life choices':'Freedom','Country or region':'Country'})

### Pruning Unnecessary 2017 Features

Dropping confidence interval bounds (`Whisker.*`) and calculation residuals (`Dystopia.Residual`) from `df1` to ensure all annual DataFrames share an identical set of core indicators.

In [3]:
df1=df1.drop(columns=['Whisker.high','Whisker.low','Dystopia.Residual'])

### Injecting Year Columns

Adding `Year` tags (2017, 2018, 2019) to each respective dataset so we can slice, group, and analyze happiness metrics over time once the files are concatenated.

In [4]:
df1['Year']=2017
df2['Year']=2018
df3['Year']=2019

### Final Concatenation & Schema Inspection

Combining all standardized annual datasets into `df` with a clean, reindexed layout. Previewing the top rows to confirm seamless alignment across all features.

In [5]:
df=pd.concat([df1,df2,df3],ignore_index=True)
print(df.shape)
df.head()

(467, 10)


,Country,Overall rank,Score,GDP per capita,Social support,Healthy life expectancy,Freedom,Generosity,Perceptions of corruption,Year
0,Norway,1,7.537,1.616463,1.533524,0.796667,0.635423,0.362012,0.315964,2017
1,Denmark,2,7.522,1.482383,1.551122,0.792566,0.626007,0.355280,0.400770,2017
2,Iceland,3,7.504,1.480633,1.610574,0.833552,0.627163,0.475540,0.153527,2017
3,Switzerland,4,7.494,1.564980,1.516912,0.858131,0.620071,0.290549,0.367007,2017
4,Finland,5,7.469,1.443572,1.540247,0.809158,0.617951,0.245483,0.382612,2017


### Data Quality & Missingness Inspection

Checking for missing values (`NaN`) and schema types across `df`. Filtering the dataset specifically for missing `Perceptions of corruption` records to inform imputation strategy.

In [6]:
print("="*20,"Number Of Null Values","="*20)
print(df.isnull().sum())

print("\n","="*20,"Columns Description","="*20)
print(df.info())

df[df['Perceptions of corruption'].isnull()]

==================== Number Of Null Values ====================
Country                      0
Overall rank                 0
Score                        0
GDP per capita               0
Social support               0
Healthy life expectancy      0
Freedom                      0
Generosity                   0
Perceptions of corruption    1
Year                         0
dtype: int64

 ==================== Columns Description ====================
<class 'pandas.DataFrame'>
RangeIndex: 467 entries, 0 to 466
Data columns (total 10 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   Country                    467 non-null    str    
 1   Overall rank               467 non-null    int64  
 2   Score                      467 non-null    float64
 3   GDP per capita             467 non-null    float64
 4   Social support             467 non-null    float64
 5   Healthy life expectancy    467 non-null    float64
 6   Fr

,Country,Overall rank,Score,GDP per capita,Social support,Healthy life expectancy,Freedom,Generosity,Perceptions of corruption,Year
174,United Arab Emirates,20,6.774,2.096,0.776,0.67,0.284,0.186,NaN,2018


### Validating Unique Country Year Pairs

Calculating the sum of duplicate `(Year, Country)` combinations in `df` to confirm that each country entry is unique per annual survey.

In [7]:
duplicated_countries=df.duplicated(subset=['Year','Country']).sum()
print(f"The duplicate countries in year 2017, 2018, and 2019 are {duplicated_countries}")

The duplicate countries in year 2017, 2018, and 2019 are 0


### Missing Value Imputation For UAE (Year 2018)

Calculating the mean `Perceptions of corruption` value for the United Arab Emirates across 2017 and 2019, then filling the missing 2018 value to restore dataset completeness.

In [8]:
year_17_19=df.loc[(df['Country']=='United Arab Emirates') & ((df['Year']==2017) | (df['Year']==2019)),'Perceptions of corruption'].mean()
df.loc[(df['Country']=='United Arab Emirates') & (df['Year']==2018),'Perceptions of corruption']=year_17_19


### Validating UAE Data Completeness

Filtering for `Country == 'United Arab Emirates'` across all survey years to confirm that the 2018 `Perceptions of corruption` score has been cleanly populated.

In [9]:
df[df['Country']=='United Arab Emirates']

,Country,Overall rank,Score,GDP per capita,Social support,Healthy life expectancy,Freedom,Generosity,Perceptions of corruption,Year
20,United Arab Emirates,21,6.648,1.626343,1.26641,0.726798,0.608345,0.360942,0.324490,2017
174,United Arab Emirates,20,6.774,2.096000,0.77600,0.670000,0.284000,0.186000,0.253245,2018
331,United Arab Emirates,21,6.825,1.503000,1.31000,0.825000,0.598000,0.262000,0.182000,2019


### Top 10 Highest Average Happiness Scores

Grouping `df` by `Country` and calculating the 3 year mean `Score` to reveal the consistently highest ranking nations between 2017 and 2019.

In [10]:
grouped_score=df.groupby('Country')['Score'].mean()
top_10=grouped_score.sort_values(ascending=False).head(10)
top_10

Country
Finland        7.623333
Norway         7.561667
Denmark        7.559000
Iceland        7.497667
Switzerland    7.487000
Netherlands    7.435333
New Zealand    7.315000
Sweden         7.313667
Canada         7.307333
Australia      7.261333
Name: Score, dtype: float64

### Top 10 Lowest Average Happiness Scores

Sorting the 3 year mean `Score` in ascending order to extract the 10 lowest ranking countries between 2017 and 2019.

In [11]:
bottom_10=grouped_score.sort_values(ascending=True).head(10)
bottom_10

Country
Central African Republic    2.953000
Burundi                     3.195000
South Sudan                 3.232667
Tanzania                    3.294333
Rwanda                      3.404333
Yemen                       3.442667
Syria                       3.462000
Afghanistan                 3.543000
Haiti                       3.594000
Botswana                    3.614667
Name: Score, dtype: float64

### Correlation Calculation For Selected Columns

Calculated correlation among different columns to check the strength and direction of the linear relationship.

In [13]:
selected_col=['Score', 'GDP per capita', 'Social support','Healthy life expectancy', 'Freedom', 'Generosity','Perceptions of corruption']
df[selected_col].corr(numeric_only=True)

,Score,GDP per capita,Social support,Healthy life expectancy,Freedom,Generosity,Perceptions of corruption
Score,1.000000,0.797052,0.758071,0.750628,0.549655,0.116574,0.407754
GDP per capita,0.797052,1.000000,0.696427,0.781642,0.344916,-0.005791,0.332820
Social support,0.758071,0.696427,1.000000,0.643913,0.422594,0.002014,0.202775
Healthy life expectancy,0.750628,0.781642,0.643913,1.000000,0.318608,-0.030055,0.270219
Freedom,0.549655,0.344916,0.422594,0.318608,1.000000,0.264407,0.453481
Generosity,0.116574,-0.005791,0.002014,-0.030055,0.264407,1.000000,0.322505
Perceptions of corruption,0.407754,0.332820,0.202775,0.270219,0.453481,0.322505,1.000000


The correlation table shows that Score has the strongest association with `GDP per capita`, `Social support`, and `Healthy life expectancy` suggesting that countries with stronger economies also tend to have better social support systems and health outcomes, and these factors move together closely with happiness. `Freedom` and `Perceptions of corruption` show a moderate **positive association** with `Score`, while `Generosity` stands out with a notably **weak correlation (0.12)** likely because it measures giving relative to income rather than income itself, making it less directly tied to overall happiness. It's worth noting that correlation does not imply causation a strong economy does not automatically guarantee **freedom, low corruption, or generosity** these are separate dimensions that a country's wealth alone doesn't determine.

### Generating Continents For Countries

Using country_converter to map each unique value in Country to its corresponding continent since the raw dataset doesn't include a region/continent column needed for the regional comparison analysis. Results are stored in `country_region_df` for merging with the main `df` so that further analysis can be done.

In [14]:
unique_countries=df['Country'].unique()
cont=coco.convert(names=unique_countries, to='continent')
country_region_df = pd.DataFrame({
    'Country': unique_countries,
    'Continent': cont
})
country_region_df.shape

(164, 2)

### Merging country_region_df & df

Performing a **left join** of `country_region_df` onto `df` using Country as the key, preserving all original rows while attaching the corresponding `Continent` for each entry. Verifying row and column counts post merge to confirm no rows were lost or duplicated

In [15]:
merging=df.merge(country_region_df, on='Country',how='left')
merging.shape

(467, 11)

### Post Merge Verification

Checking for null values in the `Continent` column to confirm every country was successfully mapped and none were left unmatched after the merge. Once verified, `merging` is assigned back to `df` as the main working dataframe, and the top 5 rows are previewed to confirm everything looks correct.

In [22]:
merging['Continent'].isnull().sum()

np.int64(0)

In [17]:
df=merging

In [18]:
df.head()

,Country,Overall rank,Score,GDP per capita,Social support,Healthy life expectancy,Freedom,Generosity,Perceptions of corruption,Year,Continent
0,Norway,1,7.537,1.616463,1.533524,0.796667,0.635423,0.362012,0.315964,2017,Europe
1,Denmark,2,7.522,1.482383,1.551122,0.792566,0.626007,0.355280,0.400770,2017,Europe
2,Iceland,3,7.504,1.480633,1.610574,0.833552,0.627163,0.475540,0.153527,2017,Europe
3,Switzerland,4,7.494,1.564980,1.516912,0.858131,0.620071,0.290549,0.367007,2017,Europe
4,Finland,5,7.469,1.443572,1.540247,0.809158,0.617951,0.245483,0.382612,2017,Europe


In [20]:
grouped_continent=df.groupby('Continent')['Score'].agg(['mean','std'])
grouped_continent

,mean,std
Continent,,
Africa,4.299571,0.655906
America,6.052268,0.763256
Asia,5.281277,0.853764
Europe,6.193375,0.902771
Oceania,7.288167,0.035227


In [21]:
df.groupby('Continent')['Country'].size()

Continent
Africa     133
America     71
Asia       137
Europe     120
Oceania      6
Name: Country, dtype: int64